# Notebook 1: Safety RM Training + Response Scoring

This notebook:
1. Trains a Safety Reward Model (RM) on the 485 preference pairs
2. Generates responses to AdvBench prompts using the SFT baseline student
3. Scores every response with the frozen RM

Output: `rm_adapter.zip` + `advbench_scored.jsonl` (used by Notebook 2 for offline RL)

**Only one model is on GPU at any time.**

In [ ]:
!pip install transformers peft accelerate bitsandbytes datasets

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import gc, json, csv, io, shutil, torch, urllib.request
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import PeftModel, LoraConfig, get_peft_model, TaskType

BASE_MODEL_ID    = "meta-llama/Llama-3.2-1B"
REPO_URL         = "https://github.com/202520030411/Distill-Safety-Aligned-Models.git"
SFT_BASELINE_DIR = "repo/outputs/baseline"
DPO_PAIRS_PATH   = "repo/data/dpo_pairs.jsonl"

RM_OUTPUT_DIR       = "/kaggle/working/outputs/rm"
SCORED_OUTPUT_PATH  = "/kaggle/working/advbench_scored.jsonl"
os.makedirs(RM_OUTPUT_DIR, exist_ok=True)

QNF4_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

REFUSAL_PREFIXES = [
    "I cannot", "I can't", "I'm not able", "I won't", "I'm unable",
    "I don't", "I apologize", "As an AI", "I'm sorry", "I must decline",
    "I'm not going to", "That's not something", "I am not able",
]

def is_refusal(text):
    t = text.lower()
    return any(p.lower() in t for p in REFUSAL_PREFIXES)

def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF_TOKEN loaded")

In [ ]:
!git clone {REPO_URL} repo
!ls repo/

## Load Data

In [ ]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f]

pairs = load_jsonl(DPO_PAIRS_PATH)
print(f"Loaded {len(pairs)} preference pairs")

def format_pair(ex):
    prompt = ex["prompt"]
    return {
        "chosen":   f"Human: {prompt}\n\nAssistant: {ex['chosen']}",
        "rejected": f"Human: {prompt}\n\nAssistant: {ex['rejected']}",
    }

rm_ds = Dataset.from_list([format_pair(p) for p in pairs])
split = rm_ds.train_test_split(test_size=0.1, seed=42)
rm_train_ds = split["train"]
rm_eval_ds  = split["test"]
print(f"RM train: {len(rm_train_ds)}  eval: {len(rm_eval_ds)}")

In [ ]:
ADVBENCH_URL = (
    "https://raw.githubusercontent.com/llm-attacks/llm-attacks/main"
    "/data/advbench/harmful_behaviors.csv"
)
with urllib.request.urlopen(ADVBENCH_URL) as resp:
    reader = csv.DictReader(io.StringIO(resp.read().decode()))
    advbench_prompts = [row["goal"] for row in reader]

print(f"AdvBench prompts: {len(advbench_prompts)}")
print("Example:", advbench_prompts[0])

## Phase 1 — Train Safety Reward Model

Only the RM is on GPU.

In [ ]:
print("=" * 60)
print("PHASE 1: Training Safety Reward Model")
print("=" * 60)

rm_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=os.environ["HF_TOKEN"])
rm_tokenizer.pad_token = rm_tokenizer.eos_token
rm_tokenizer.padding_side = "right"

rm_base = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_ID,
    num_labels=1,
    torch_dtype=torch.float32,
    device_map={"":0},
    token=os.environ["HF_TOKEN"],
)
rm_base.config.pad_token_id = rm_tokenizer.pad_token_id

rm_lora = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    modules_to_save=["score"],
    lora_dropout=0.05, bias="none", task_type=TaskType.SEQ_CLS,
)
rm_model = get_peft_model(rm_base, rm_lora)
rm_model.print_trainable_parameters()

In [ ]:
import torch.nn.functional as F

NUM_EPOCHS = 2
GRAD_ACCUM = 8
LR = 5e-5
MAX_LEN = 256
MAX_GRAD_NORM = 1.0

optimizer = torch.optim.AdamW(
    [p for p in rm_model.parameters() if p.requires_grad], lr=LR
)

rm_model.train()
global_step = 0
accum_loss = 0.0

print(f"Starting manual RM training: {len(rm_train_ds)} pairs, "
      f"{NUM_EPOCHS} epochs, grad_accum={GRAD_ACCUM}")

for epoch in range(NUM_EPOCHS):
    indices = torch.randperm(len(rm_train_ds)).tolist()
    for i, idx in enumerate(indices):
        row = rm_train_ds[idx]

        chosen_tok = rm_tokenizer(
            row["chosen"], return_tensors="pt",
            truncation=True, max_length=MAX_LEN, padding="max_length",
        ).to(rm_model.device)
        rejected_tok = rm_tokenizer(
            row["rejected"], return_tensors="pt",
            truncation=True, max_length=MAX_LEN, padding="max_length",
        ).to(rm_model.device)

        r_chosen  = rm_model(**chosen_tok).logits[0, 0].float()
        r_rejected = rm_model(**rejected_tok).logits[0, 0].float()
        loss = -F.logsigmoid(r_chosen - r_rejected) / GRAD_ACCUM

        loss.backward()
        accum_loss += loss.item()

        if (i + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(rm_model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1
            print(f"  epoch {epoch+1}/{NUM_EPOCHS}  step {global_step}  "
                  f"loss={accum_loss:.4f}  r_chosen={r_chosen.item():.3f}  "
                  f"r_rejected={r_rejected.item():.3f}")
            accum_loss = 0.0

    optimizer.step()
    optimizer.zero_grad()
    print(f"Epoch {epoch+1} done.")

rm_model.save_pretrained(RM_OUTPUT_DIR)
rm_tokenizer.save_pretrained(RM_OUTPUT_DIR)
print(f"RM saved to {RM_OUTPUT_DIR}")

del optimizer, rm_model, rm_base
free_gpu()
print("GPU freed after RM training")

## Phase 2 — Generate Responses with SFT Baseline

Only the SFT generation model is on GPU. RM is not loaded.

In [ ]:
print("=" * 60)
print("PHASE 2: Generating responses to AdvBench prompts")
print("=" * 60)

gen_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=os.environ["HF_TOKEN"])
gen_tokenizer.pad_token = gen_tokenizer.eos_token
gen_tokenizer.padding_side = "left"

gen_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map={"":0},
    token=os.environ["HF_TOKEN"],
)
gen_model = PeftModel.from_pretrained(gen_base, SFT_BASELINE_DIR)
gen_model = gen_model.merge_and_unload()
gen_model.eval()
print("SFT baseline loaded and merged")

In [ ]:
NUM_GENERATIONS = 4
PROMPT_TEMPLATE = "Human: {}\n\nAssistant:"
responses_path = "/kaggle/working/advbench_responses.jsonl"

with open(responses_path, "w") as fout:
    for i, prompt in enumerate(advbench_prompts):
        full_prompt = PROMPT_TEMPLATE.format(prompt)
        inputs = gen_tokenizer(
            full_prompt, return_tensors="pt", truncation=True, max_length=256,
        ).to(gen_model.device)

        with torch.no_grad():
            outputs = gen_model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                num_return_sequences=NUM_GENERATIONS,
                pad_token_id=gen_tokenizer.pad_token_id,
            )

        responses = []
        for seq in outputs:
            text = gen_tokenizer.decode(seq[inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            responses.append(text.strip())

        fout.write(json.dumps({"prompt": prompt, "responses": responses}) + "\n")

        if (i + 1) % 50 == 0:
            print(f"  Generated {i+1}/{len(advbench_prompts)} prompts")

print(f"Done. Saved {len(advbench_prompts)} x {NUM_GENERATIONS} responses to {responses_path}")

del gen_model, gen_base
free_gpu()
print("GPU freed after generation")

## Phase 3 — Score Responses with Frozen RM

Only the frozen RM is on GPU. Generation model is not loaded.

In [ ]:
print("=" * 60)
print("PHASE 3: Scoring responses with frozen RM")
print("=" * 60)

score_base = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_ID,
    num_labels=1,
    torch_dtype=torch.float16,
    device_map={"":0},
    token=os.environ["HF_TOKEN"],
)
score_base.config.pad_token_id = rm_tokenizer.pad_token_id
score_model = PeftModel.from_pretrained(score_base, RM_OUTPUT_DIR)
score_model.eval()
for p in score_model.parameters():
    p.requires_grad = False
print("Frozen RM loaded for scoring")

In [ ]:
generated = load_jsonl(responses_path)
print(f"Scoring {len(generated)} prompts x {NUM_GENERATIONS} responses...")

with open(SCORED_OUTPUT_PATH, "w") as fout:
    for i, row in enumerate(generated):
        prompt = row["prompt"]
        rewards = []
        for resp in row["responses"]:
            full_text = f"Human: {prompt}\n\nAssistant: {resp}"
            inputs = rm_tokenizer(
                full_text, return_tensors="pt",
                truncation=True, max_length=256, padding=True,
            ).to(score_model.device)

            with torch.no_grad():
                rm_score = score_model(**inputs).logits[0, 0].item()

            heuristic = 0.5 if is_refusal(resp) else -0.5
            rewards.append(round(rm_score + heuristic, 4))

        fout.write(json.dumps({
            "prompt": prompt,
            "responses": row["responses"],
            "rewards": rewards,
        }) + "\n")

        if (i + 1) % 100 == 0:
            print(f"  Scored {i+1}/{len(generated)}")

print(f"Done. Saved scored data to {SCORED_OUTPUT_PATH}")

del score_model, score_base
free_gpu()

In [ ]:
# Show sample output
sample = load_jsonl(SCORED_OUTPUT_PATH)[:3]
for s in sample:
    print(f"Prompt: {s['prompt'][:60]}...")
    for r, rw in zip(s['responses'], s['rewards']):
        tag = "REFUSE" if is_refusal(r) else "COMPLY"
        print(f"  [{tag}] reward={rw:.3f}  {r[:60]}...")
    print()

In [ ]:
shutil.make_archive("/kaggle/working/rm_adapter", "zip", RM_OUTPUT_DIR)
shutil.copy(SCORED_OUTPUT_PATH, "/kaggle/working/advbench_scored.jsonl")
print("Download rm_adapter.zip and advbench_scored.jsonl from the Output tab.")